# EDA: Motos eléctricas procesadas (300 anuncios)

Fuentes: `data/processed/final/motos_completos.json` + `motos_parciales.json`.
Solo estadísticas y distribuciones; sin gráficos todavía.

In [1]:
import json
import pandas as pd
from pathlib import Path

BASE = None
for nivel in ("data/processed/final", "../data/processed/final", "../../data/processed/final"):
    if Path(nivel).exists():
        BASE = Path(nivel)
        break
assert BASE is not None, "No se encontró la carpeta de finales"

with open(f"{BASE}/motos_completos.json", encoding="utf-8") as f:
    completos = json.load(f)
with open(f"{BASE}/motos_parciales.json", encoding="utf-8") as f:
    parciales = json.load(f)

df_motos = pd.DataFrame(completos + parciales)
df_motos["tipo_registro"] = ["completo"] * len(completos) + ["parcial"] * len(parciales)
print(f"Total anuncios: {len(df_motos)}")
print(f"Completos: {len(completos)} | Parciales: {len(parciales)}")

Total anuncios: 300
Completos: 83 | Parciales: 217


In [2]:
nulos = df_motos.isnull().mean().mul(100).round(1)
nulos_df = pd.DataFrame({"campo": nulos.index, "%_nulos": nulos.values})
nulos_df

,campo,%_nulos
0,id_anuncio,0.0
1,titulo,0.0
2,precio_usd,0.0
3,fecha_publicacion,0.0
4,marca,30.0
5,modelo,54.7
6,tipo_vehiculo,0.0
7,motor_w,50.0
8,voltaje_v,47.3
9,amperaje_ah,44.7


In [3]:
cols_num = ["precio_usd", "motor_w", "voltaje_v", "amperaje_ah", "autonomia_km", "velocidad_max_kmh", "peso_kg"]
df_motos[cols_num].describe().T

,count,mean,std,min,25%,50%,75%,max
precio_usd,300.0,2088.420000,839.728908,1.0,1500.0,2450.0,2700.0,4200.0
motor_w,150.0,2303.800000,1077.863702,350.0,1000.0,3000.0,3000.0,5000.0
voltaje_v,158.0,67.822785,9.853396,36.0,72.0,72.0,72.0,96.0
amperaje_ah,166.0,40.355422,14.794354,10.0,30.0,45.0,50.0,80.0
autonomia_km,259.0,98.583012,47.499285,0.0,70.0,120.0,120.0,200.0
velocidad_max_kmh,122.0,61.983607,16.237892,25.0,50.0,60.0,70.0,90.0
peso_kg,3.0,86.000000,26.514147,60.0,72.5,85.0,99.0,113.0


In [4]:
top_marcas = df_motos["marca"].value_counts().head(10)
top_marcas

marca
Topmaq      54
Vedca       26
Mountain    17
Izuki       16
Yoazaki     16
Bucatti     11
Fly         10
Kvitova      8
Maf          7
Xmox         6
Name: count, dtype: int64

In [5]:
top_municipios = df_motos["municipio"].value_counts().head(10)
top_municipios

municipio
Playa              72
Centro Habana      70
Plaza              50
Cerro              28
Diez de Octubre    17
La Lisa            14
Marianao           12
Boyeros             9
Habana Vieja        9
Arroyo Naranjo      8
Name: count, dtype: int64

In [6]:
dist_bateria = df_motos["tipo_bateria"].value_counts(dropna=False)
dist_bateria

tipo_bateria
NaN      177
litio    123
Name: count, dtype: int64

In [7]:
dist_estado = df_motos["estado"].value_counts(dropna=False)
dist_estado

estado
NaN      256
Nueva     44
Name: count, dtype: int64

In [8]:
con_papeles = df_motos["papeles"].eq("en regla").sum()
print(f"Anuncios con papeles 'en regla': {con_papeles}")

Anuncios con papeles 'en regla': 121


In [9]:
con_transporte = df_motos["transporte_incluido"].sum()
print(f"Anuncios con transporte incluido: {con_transporte}")

Anuncios con transporte incluido: 134


# Análisis temporal (fecha_publicacion = updatedOnByUser)

In [10]:
df_motos["fecha_publicacion"] = pd.to_datetime(df_motos["fecha_publicacion"])
df_motos["mes"] = df_motos["fecha_publicacion"].dt.to_period("M")
df_motos["dia_semana"] = df_motos["fecha_publicacion"].dt.day_name()

print("Rango de fechas (mínima y máxima):")
print("  mínima:", df_motos["fecha_publicacion"].min())
print("  máxima:", df_motos["fecha_publicacion"].max())

print("\nDistribución de anuncios por mes:")
print(df_motos["mes"].value_counts().sort_index())

print("\nPrecio medio y mediano por mes:")
print(df_motos.groupby("mes")["precio_usd"].agg(["mean", "median"]))

print("\nDistribución de anuncios por día de la semana:")
print(df_motos["dia_semana"].value_counts())

Rango de fechas (mínima y máxima):
  mínima: 2026-09-24 15:17:15.107000+00:00
  máxima: 2026-10-08 14:20:02.059000+00:00

Distribución de anuncios por mes:
mes
2026-09      1
2026-10    299
Freq: M, Name: count, dtype: int64

Precio medio y mediano por mes:
                mean  median
mes                         
2026-09  3000.000000  3000.0
2026-10  2085.371237  2450.0

Distribución de anuncios por día de la semana:
dia_semana
Wednesday    221
Tuesday       45
Thursday      34
Name: count, dtype: int64


/tmp/ipykernel_40792/3421882662.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_motos["mes"] = df_motos["fecha_publicacion"].dt.to_period("M")
